In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"C:\Users\bonga\OneDrive\Desktop\revenue-leakage-analytic\reports\anomaly_investigation_report.csv")
df.sample(10)

,order_id,product_id,discount_pct,net_sales,estimated_cost,gross_profit,profit_margin_pct,anomaly_score,investigation_reason
721,O0019182,P00860,1.85,57726.66,36378.87,21347.79,36.980816,0.061466,Unusual transaction pattern - investigate
1642,O0008165,P00765,20.53,53405.65,43408.44,9997.21,18.719386,0.039096,Unusual transaction pattern - investigate
1132,O0095211,P00592,53.30,6334.39,11453.81,-5119.42,-80.819463,0.049946,High discount with low margin
1984,O0052089,P00981,54.05,595.03,1173.20,-578.17,-97.166529,0.030877,High discount with low margin
985,O0020424,P00393,60.77,7166.16,13073.23,-5907.07,-82.430060,0.053297,High discount with low margin
2049,O0078356,P00191,2.17,50521.72,32692.14,17829.58,35.290920,0.029288,Unusual transaction pattern - investigate
3068,O0034366,P00242,41.66,8219.63,13198.04,-4978.41,-60.567325,0.006743,High discount with low margin
3480,O0084243,P00992,43.70,5540.26,8913.99,-3373.73,-60.894796,0.000358,High discount with low margin
3109,O0015316,P00889,46.23,2095.80,3673.04,-1577.24,-75.257181,0.005994,High discount with low margin
2517,O0019891,P00473,50.78,569.84,1058.50,-488.66,-85.753896,0.017516,High discount with low margin


In [3]:

import pandas as pd
import numpy as np

def recommend_action(row):
    discount = pd.to_numeric(
        row.get("discount_pct", np.nan), errors="coerce"
    )
    margin = pd.to_numeric(
        row.get("profit_margin_pct", np.nan), errors="coerce"
    )
    profit = pd.to_numeric(
        row.get("gross_profit", np.nan), errors="coerce"
    )

    # Rule 1: Negative gross profit
    if pd.notna(profit) and profit < 0:
        return (
            "HIGH",
            "Negative gross profit",
            "Validate product cost, quantity and pricing."
        )

    # Rule 2: High discount combined with low profit margin
    if (
        pd.notna(discount)
        and discount > 40
        and pd.notna(margin)
        and margin < 5
    ):
        return (
            "HIGH",
            "High discount with low margin",
            "Review discount limits and product profitability."
        )

    # Rule 3: High discount without the preceding HIGH-risk conditions
    if pd.notna(discount) and discount > 40:
        return (
            "MEDIUM",
            "High discount",
            "Check campaign performance and discount effectiveness."
        )

    # Rule 4: Other unusual transactions requiring investigation
    return (
        "REVIEW",
        "Unusual transaction pattern",
        "Investigate the transaction before taking action."
    )


In [4]:
results = df.apply(recommend_action, axis=1, result_type="expand")

In [5]:
results.columns = [
    "risk_level",
    "business_issue",
    "recommended_action"
]

In [6]:
df = pd.concat([df, results], axis=1)

In [7]:
df.to_csv("final_business_recommendations.csv", index=False)

In [8]:
print(df["risk_level"].value_counts())
print("\nSample recommendations:")
print(df[["order_id", "risk_level", "business_issue","recommended_action"]].sample(10).to_string(index=False))

risk_level
HIGH      2450
REVIEW    1047
Name: count, dtype: int64

Sample recommendations:
order_id risk_level              business_issue                                recommended_action
O0085395       HIGH       Negative gross profit      Validate product cost, quantity and pricing.
O0056385       HIGH       Negative gross profit      Validate product cost, quantity and pricing.
O0024331     REVIEW Unusual transaction pattern Investigate the transaction before taking action.
O0072070       HIGH       Negative gross profit      Validate product cost, quantity and pricing.
O0034299     REVIEW Unusual transaction pattern Investigate the transaction before taking action.
O0047576       HIGH       Negative gross profit      Validate product cost, quantity and pricing.
O0094292       HIGH       Negative gross profit      Validate product cost, quantity and pricing.
O0052633       HIGH       Negative gross profit      Validate product cost, quantity and pricing.
O0060188       HIGH       